#### Importing Libraries

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle

#### Load the Dataset

In [4]:
data = pd.read_csv("Churn_Modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


#### Preprocessing the data
Drop irrelevant columns

In [5]:
data = data.drop(['RowNumber','CustomerId','Surname'],axis=1)


In [6]:
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


#### Encode categorical variables
Changing Gender column values from "Male" & "Female" to 0,1. It will take values as classes. If Male and Female are there, 2 classes= 0(female),1(male). If multiple classes like cat, bird, dog are there, it will take it as 0(bird),1(cat),2(dog). Use alphabetical order for internal mapping

In [7]:
label_endoder_gender = LabelEncoder()
data['Gender']=label_endoder_gender.fit_transform(data['Gender'])
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,1,39,5,0.00,2,1,0,96270.64,0
9996,516,France,1,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,0,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,1,42,3,75075.31,2,1,0,92888.52,1


#### Geography multiclass column Onehot encoding
By using onehot encoding we are going to assign values for geography column. We don't use label encoding here because if 0,1,2 are there, model might think that label 2 is greater than 0,1 labels, which is wrong. So we will be using onehot encoding and sparse matrix to make it into sparse matrix and keep it as 3 different columns

In [8]:
from sklearn.preprocessing import OneHotEncoder
onehot_encoder_geo=OneHotEncoder()
geo_encoder=onehot_encoder_geo.fit_transform(data[['Geography']])
geo_encoder

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [9]:
onehot_encoder_geo.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [10]:
geo_encoder.toarray()

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]], shape=(10000, 3))

In [11]:
geo_encoded_df=pd.DataFrame(geo_encoder.toarray(),columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

In [12]:
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


#### Combining onehot encoder columns with the original data
First remove Geography column and then add the encoded columns


In [13]:
data=pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


#### Save the encoders and scaler

In [17]:
with open('label_endoder_gender.pkl','wb') as file:
    pickle.dump(label_endoder_gender,file)


In [15]:
with open('onehot_encoder_geo.pkl','wb') as file:
    pickle.dump(onehot_encoder_geo,file)

#### Divide the dataset into independent and dependent features 

In [16]:
X=data.drop('Exited',axis=1)
y=data['Exited']

#### Split the data in training and testing sets
test_size=0.2 means 20% data for testing and remaining 80% for training.
And random_state=42, is like machine/model shuffles your data everytime we run the script. If we keep some random seed number 42, everytime it shuffles, you'll get the same sequence. Even if it's run on other machine by other user, it will shuffle with seed number 42. Shuffling on a certain number with algorithms is completely dealt by machine, we just make sure that we use consistent random_state so that who ever uses, they will get the same shuffling of datasets. Not like someone got better train/test dataset.


In [18]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=29)

#### Scale these features using Standard Scaler
Standard scaler is to ensure that columns with large numeric values don't dominate columns with small values.
If you were to accidentally use fit_transform on X_test, you would cause a major issue called Data Leakage.In the real world, your test set represents future, unseen data (like a new customer signing up tomorrow). Your model cannot know the average age or credit score of future customers ahead of time. If you calculate a brand-new mean from X_test, your model is subtly "cheating" by gaining information about the test set's distribution before making predictions.By running only transform() on X_test, you ensure your test set remains completely pure and isolated, giving you an honest evaluation of your model's true performance.

Standardization can be done with mathematical formula.

In [19]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [20]:
with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)

In [21]:
X_train

array([[-0.05801516,  0.91855473, -1.23709023, ..., -0.99675526,
        -0.57965968,  1.73147365],
       [-1.34461817, -1.08866675,  0.6683424 , ...,  1.0032553 ,
        -0.57965968, -0.57754272],
       [ 2.04827848,  0.91855473, -0.76073207, ..., -0.99675526,
        -0.57965968,  1.73147365],
       ...,
       [-0.53530338,  0.91855473,  1.33524382, ..., -0.99675526,
         1.72515018, -0.57754272],
       [-0.48342422,  0.91855473, -0.37964554, ..., -0.99675526,
         1.72515018, -0.57754272],
       [-1.63514143, -1.08866675, -0.09383065, ..., -0.99675526,
         1.72515018, -0.57754272]], shape=(8000, 12))

#### ANN Implementation

In [22]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

#### Build our ANN Model

In [23]:
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)), ## HL1 Connected with input layer
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid') ##output layer
])

In [25]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [26]:
import tensorflow
opt= tensorflow.keras.optimizers.Adam(learning_rate=0.01)

#### Compile the model

In [29]:
model.compile(optimizer=opt, loss="binary_crossentropy", metrics=['accuracy'])